In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from typing import Tuple, List
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Matplotlib version: {matplotlib.__version__}")
print(f"Pandas version: {pd.__version__}")


In [ ]:
class SimpleRNN:
    """
    A simple RNN implementation from scratch using NumPy
    """
    
    def __init__(self, input_size: int, hidden_size: int, output_size: int, learning_rate: float = 0.01):
        """
        Initialize RNN parameters
        
        Args:
            input_size: Size of input vector
            hidden_size: Size of hidden state
            output_size: Size of output vector
            learning_rate: Learning rate for training
        """
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.learning_rate = learning_rate
        
        # Initialize weights with Xavier initialization
        self.W_xh = np.random.randn(hidden_size, input_size) * np.sqrt(2.0 / input_size)
        self.W_hh = np.random.randn(hidden_size, hidden_size) * np.sqrt(2.0 / hidden_size)
        self.W_hy = np.random.randn(output_size, hidden_size) * np.sqrt(2.0 / hidden_size)
        
        # Initialize biases
        self.b_h = np.zeros((hidden_size, 1))
        self.b_y = np.zeros((output_size, 1))
        
        # Initialize hidden state
        self.h = np.zeros((hidden_size, 1))
        
        print(f"RNN initialized with:")
        print(f"Input size: {input_size}")
        print(f"Hidden size: {hidden_size}")
        print(f"Output size: {output_size}")
        print(f"Learning rate: {learning_rate}")
    
    def forward(self, inputs: np.ndarray) -> Tuple[List[np.ndarray], List[np.ndarray]]:
        """
        Forward pass through the RNN
        
        Args:
            inputs: Input sequence of shape (seq_len, input_size, 1)
            
        Returns:
            Tuple of (hidden_states, outputs)
        """
        seq_len = len(inputs)
        hidden_states = []
        outputs = []
        
        # Reset hidden state
        h = np.zeros((self.hidden_size, 1))
        
        for t in range(seq_len):
            # Get input at time t
            x_t = inputs[t].reshape(-1, 1)
            
            # Compute new hidden state
            h = np.tanh(self.W_xh @ x_t + self.W_hh @ h + self.b_h)
            
            # Compute output
            y_t = self.W_hy @ h + self.b_y
            
            hidden_states.append(h.copy())
            outputs.append(y_t.copy())
        
        return hidden_states, outputs
    
    def backward(self, inputs: np.ndarray, targets: np.ndarray, 
                hidden_states: List[np.ndarray], outputs: List[np.ndarray]) -> float:
        """
        Backward pass through the RNN (Backpropagation Through Time)
        
        Args:
            inputs: Input sequence
            targets: Target sequence
            hidden_states: Hidden states from forward pass
            outputs: Outputs from forward pass
            
        Returns:
            Total loss
        """
        seq_len = len(inputs)
        
        # Initialize gradients
        dW_xh = np.zeros_like(self.W_xh)
        dW_hh = np.zeros_like(self.W_hh)
        dW_hy = np.zeros_like(self.W_hy)
        db_h = np.zeros_like(self.b_h)
        db_y = np.zeros_like(self.b_y)
        
        # Initialize hidden state gradient
        dh_next = np.zeros((self.hidden_size, 1))
        
        total_loss = 0
        
        # Backward through time
        for t in reversed(range(seq_len)):
            # Get current inputs and targets
            x_t = inputs[t].reshape(-1, 1)
            y_t = targets[t].reshape(-1, 1)
            
            # Output loss and gradient
            dy = outputs[t] - y_t
            total_loss += 0.5 * np.sum(dy ** 2)
            
            # Output layer gradients
            dW_hy += dy @ hidden_states[t].T
            db_y += dy
            
            # Hidden state gradient
            dh = self.W_hy.T @ dy + dh_next
            
            # Gradient through tanh
            dh_raw = (1 - hidden_states[t] ** 2) * dh
            
            # Weight gradients
            dW_xh += dh_raw @ x_t.T
            db_h += dh_raw
            
            if t > 0:
                dW_hh += dh_raw @ hidden_states[t-1].T
                dh_next = self.W_hh.T @ dh_raw
            else:
                dh_next = np.zeros((self.hidden_size, 1))
        
        # Clip gradients to prevent exploding gradients
        for grad in [dW_xh, dW_hh, dW_hy, db_h, db_y]:
            np.clip(grad, -5, 5, out=grad)
        
        # Update weights
        self.W_xh -= self.learning_rate * dW_xh
        self.W_hh -= self.learning_rate * dW_hh
        self.W_hy -= self.learning_rate * dW_hy
        self.b_h -= self.learning_rate * db_h
        self.b_y -= self.learning_rate * db_y
        
        return total_loss / seq_len

# Test the RNN class
print("SimpleRNN class defined successfully!")


In [ ]:
# Create synthetic data for testing our RNN
def generate_sine_wave_data(seq_length: int = 50, num_sequences: int = 1000) -> Tuple[np.ndarray, np.ndarray]:
    """
    Generate sine wave data for testing RNN
    
    Args:
        seq_length: Length of each sequence
        num_sequences: Number of sequences to generate
        
    Returns:
        Tuple of (inputs, targets)
    """
    inputs = []
    targets = []
    
    for _ in range(num_sequences):
        # Random frequency and phase
        freq = np.random.uniform(0.1, 0.5)
        phase = np.random.uniform(0, 2 * np.pi)
        
        # Generate time steps
        t = np.linspace(0, 4 * np.pi, seq_length + 1)
        
        # Generate sine wave
        sequence = np.sin(freq * t + phase)
        
        # Input is sequence[:-1], target is sequence[1:]
        inputs.append(sequence[:-1])
        targets.append(sequence[1:])
    
    return np.array(inputs), np.array(targets)

# Generate test data
print("Generating synthetic sine wave data...")
X_train, y_train = generate_sine_wave_data(seq_length=20, num_sequences=500)
X_test, y_test = generate_sine_wave_data(seq_length=20, num_sequences=100)

print(f"Training data shape: {X_train.shape}")
print(f"Training targets shape: {y_train.shape}")
print(f"Test data shape: {X_test.shape}")
print(f"Test targets shape: {y_test.shape}")

# Visualize some sample data
plt.figure(figsize=(12, 4))
for i in range(3):
    plt.subplot(1, 3, i+1)
    plt.plot(X_train[i], label='Input', marker='o', markersize=3)
    plt.plot(y_train[i], label='Target', marker='s', markersize=3)
    plt.title(f'Sample {i+1}')
    plt.legend()
    plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# Train and test the RNN
def train_rnn(rnn, X_train, y_train, epochs=100):
    """
    Train the RNN on the given data
    
    Args:
        rnn: RNN model to train
        X_train: Training inputs
        y_train: Training targets
        epochs: Number of training epochs
        
    Returns:
        List of losses during training
    """
    losses = []
    
    for epoch in range(epochs):
        epoch_loss = 0
        
        # Train on each sequence
        for i in range(len(X_train)):
            # Convert to list of inputs
            inputs = [X_train[i][t:t+1] for t in range(len(X_train[i]))]
            targets = [y_train[i][t:t+1] for t in range(len(y_train[i]))]
            
            # Forward and backward pass
            hidden_states, outputs = rnn.forward(inputs)
            loss = rnn.backward(inputs, targets, hidden_states, outputs)
            epoch_loss += loss
        
        # Average loss for the epoch
        avg_loss = epoch_loss / len(X_train)
        losses.append(avg_loss)
        
        # Print progress
        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch + 1}/{epochs}, Loss: {avg_loss:.6f}")
    
    return losses

# Initialize and train the RNN
print("Initializing RNN...")
rnn = SimpleRNN(input_size=1, hidden_size=10, output_size=1, learning_rate=0.01)

print("\nTraining RNN...")
training_losses = train_rnn(rnn, X_train, y_train, epochs=100)

# Plot training loss
plt.figure(figsize=(10, 6))
plt.plot(training_losses)
plt.title('RNN Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.show()

print(f"Final training loss: {training_losses[-1]:.6f}")


In [ ]:
# Test the trained RNN and visualize predictions
def test_rnn(rnn, X_test, y_test):
    """
    Test the trained RNN and calculate metrics
    
    Args:
        rnn: Trained RNN model
        X_test: Test inputs
        y_test: Test targets
        
    Returns:
        Test loss and predictions
    """
    total_loss = 0
    predictions = []
    
    for i in range(len(X_test)):
        # Convert to list of inputs
        inputs = [X_test[i][t:t+1] for t in range(len(X_test[i]))]
        targets = [y_test[i][t:t+1] for t in range(len(y_test[i]))]
        
        # Forward pass only
        hidden_states, outputs = rnn.forward(inputs)
        
        # Calculate loss
        loss = 0
        for t in range(len(outputs)):
            loss += 0.5 * np.sum((outputs[t] - targets[t]) ** 2)
        total_loss += loss / len(outputs)
        
        # Store predictions
        pred_sequence = [output[0, 0] for output in outputs]
        predictions.append(pred_sequence)
    
    avg_loss = total_loss / len(X_test)
    return avg_loss, predictions

# Test the RNN
print("Testing RNN...")
test_loss, predictions = test_rnn(rnn, X_test, y_test)
print(f"Test loss: {test_loss:.6f}")

# Visualize predictions vs actual
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i in range(6):
    row = i // 3
    col = i % 3
    
    axes[row, col].plot(y_test[i], label='Actual', marker='o', markersize=3)
    axes[row, col].plot(predictions[i], label='Predicted', marker='s', markersize=3)
    axes[row, col].set_title(f'Test Sample {i+1}')
    axes[row, col].legend()
    axes[row, col].grid(True)

plt.tight_layout()
plt.show()

# Calculate additional metrics
mse = np.mean([(np.array(predictions[i]) - y_test[i])**2 for i in range(len(predictions))])
mae = np.mean([np.abs(np.array(predictions[i]) - y_test[i]) for i in range(len(predictions))])

print(f"\nTest Metrics:")
print(f"Mean Squared Error: {mse:.6f}")
print(f"Mean Absolute Error: {mae:.6f}")
